In [1]:
import re
import pandas as pd
from difflib import SequenceMatcher

In [2]:
def clean(name):
    """Normalize a player name so formatting differences don't block a match."""
    name = name.lower()
    name = re.sub(r"[.\']", "", name)                    # remove periods and apostrophes
    name = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b", "", name)  # remove suffixes
    return re.sub(r"\s+", " ", name).strip()             # collapse extra spaces

In [3]:
def sim(a, b):
    """Return 0 to 1 for how alike two strings are."""
    return SequenceMatcher(None, a, b).ratio()

In [4]:
def fuzzy_match(row, wide):
    """For one unmatched portal player, find the best same-last-name candidate
    on the roster of the school he transferred to."""
    pool = wide[wide["team"] == row["destination"]]
    parts = row["name_clean"].split()
    first, last = parts[0], parts[-1]
    best, best_score = None, 0
    for _, p in pool.iterrows():
        pp = p["name_clean"].split()
        if pp[-1] != last:                 # last name must match exactly
            continue
        score = sim(first, pp[0])
        if pp[0].startswith(first) or first.startswith(pp[0]):
            score = max(score, 0.85)       # catches ben/benjamin
        if score > best_score:
            best, best_score = p["name_clean"], score
    return pd.Series([best, best_score])

In [5]:
accepted = {
    "vandrevius jacobs": "vandrevious jacobs",
    "tiaoalii savea": "tiaoalil savea",
    "yam banks": "yamarus banks",
    "benjamin yurosek": "ben yurosek",
    "dez malone": "dezjhon malone",
    "josh celiscar": "joshua celiscar",
    # Add these only if you checked position and confirmed them:
    # "lt overton": "lebbeus overton",
    # "bj mayes": "brian mayes",
    # "chip trayanum": "deamonte trayanum",

_IncompleteInputError: incomplete input (96092729.py, line 11)

In [6]:
accepted = {
    "vandrevius jacobs": "vandrevious jacobs",
    "tiaoalii savea": "tiaoalil savea",
    "yam banks": "yamarus banks",
    "benjamin yurosek": "ben yurosek",
    "dez malone": "dezjhon malone",
    "josh celiscar": "joshua celiscar",
    # Add these only if you checked position and confirmed them:
    # "lt overton": "lebbeus overton",
    # "bj mayes": "brian mayes",
    # "chip trayanum": "deamonte trayanum",
}

In [7]:
print(clean("C.J. Daniels"))        # expect: cj daniels
print(clean("Corey Flagg Jr."))     # expect: corey flagg
print(round(sim("vandrevius", "vandrevious"), 2))   # expect: about 0.95

cj daniels
corey flagg
0.95
